# **Proyecto Etapa 3 — Aprendizaje Supervisado y No Supervisado con PySpark**

### **Curso: TC5057 · Análisis de Grandes Volúmenes de Datos**
#### **Tecnológico de Monterrey**
##### **Profesor Titular: Dr. Iván Olmos Pineda**

---

### **Equipo #17**
#### **Tutor: José Carlos Soto**

| Nombre | Matrícula |
|--------|-----------|
| Diego Falcón Costilla | A00000000 |

---

**Dataset:** GTEx Analysis V10 — Gene Expression TPM (NIH / Broad Institute)  
**Archivo:** `GTEx_Analysis_2022-06-06_v10_RNASeQCv2.4.2_gene_tpm_non_lcm.gct`  
**Genes:** 59,033 | **Muestras RNASEQ:** 19,788 | **Donantes:** 981

---
## 1. Construcción de la muestra M

### 1.1 Definición de M

La muestra M se define como el conjunto de **todas** las muestras RNASEQ disponibles en GTEx V10, organizadas en las **10 particiones** derivadas de las variables de caracterización:

$$M = \{M_i : M_i \text{ es una partición de TISSUE\_GROUP} \times \text{SEX\_LABEL}\},\quad i = 1, \ldots, 10$$

con 5 grupos de tejido × 2 sexos biológicos como ejes de particionamiento.

### 1.2 Evaluación de representatividad y mejoras implementadas

Se evaluó si la muestra M es suficientemente representativa de la población P (donantes adultos del proyecto GTEx V10 con datos RNASEQ). El análisis identificó los siguientes ajustes necesarios respecto a versiones previas de trabajo:

| Aspecto evaluado | Versión previa | Versión actual | Justificación |
|-----------------|----------------|----------------|---------------|
| **Cobertura de muestras** | Sub-muestra reducida (1/100 de columnas) | Todas las 19,788 muestras RNASEQ | La tarea exige trabajar con la muestra lo más completa posible |
| **Selección de genes** | Sub-selección aleatoria de columnas | Top-500 genes por varianza inter-muestras | Los genes de mayor varianza son los más discriminantes biológicamente (Law et al., 2016) |
| **División train/test** | Aleatoria por muestra (sesgo intra-donante) | Por donante (`SUBJID`) — garantiza `Tri ∩ Tsi = ∅` | Evita que el modelo aprenda perfiles individuales en lugar de patrones generalizables |
| **Cobertura de particiones** | 2–4 particiones | 10 particiones completas | M debe representar toda la diversidad tisular de P |
| **SMTSD disponible** | No incluido | Incluido en metadatos | Permite clasificación fina por sub-tipo de tejido si se requiere |

**Conclusión de representatividad:** la muestra M con 19,616 muestras cubre los 946 donantes únicos disponibles, representa los 5 grupos de tejido y ambos sexos biológicos, y selecciona los genes de mayor varianza para maximizar la señal discriminante. Se considera suficientemente representativa para el análisis de aprendizaje automático de esta etapa.

### 1.3 Estrategia de carga del dataset

El archivo TPM tiene **59,033 genes × 19,788 muestras** (~4.7 GB). Se aplica una estrategia en dos pasos para mantener factibilidad computacional sin perder representatividad biológica:

1. **Cálculo de varianza por bloques:** lectura chunked (500 genes/bloque) con `pandas.read_csv`, ~80 MB pico por bloque. Varianza calculada de forma vectorizada con `DataFrame.var(axis=1)`.
2. **Carga de la matriz final:** solo las 500 filas de los top genes, para todas las 19,616 muestras → 78.5 MB en memoria.

Los metadatos (particiones, donantes, sexo) se cargan y procesan con **PySpark** para aprovechar el procesamiento distribuido en la unión y filtrado de los 19,788 registros.

In [1]:
import sys, os

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

_conda_env = os.path.dirname(sys.executable)
_java_home = os.path.join(_conda_env, 'Library', 'lib', 'jvm')
if os.path.isdir(_java_home):
    os.environ['JAVA_HOME'] = _java_home

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType

sys.path.insert(0, os.path.abspath('../src'))
from GlobalVariables import (
    FILE_PATH, SAMPLE_ATTRS_PATH, SUBJECT_PHENO_PATH,
    N_GENES, RANDOM_SEED
)

import random
import numpy as np
import pandas as pd
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'JAVA_HOME: {os.environ.get("JAVA_HOME", "ERROR")}')
print(f'Semilla aleatoria: {RANDOM_SEED}')
print(f'Genes en el dataset: {N_GENES:,}')

JAVA_HOME: C:\Users\diego\anaconda3\envs\big-data\Library\lib\jvm
Semilla aleatoria: 42
Genes en el dataset: 59,033


In [2]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('GTEx_Etapa3_TC5057') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print(f'Workers: {spark.sparkContext.defaultParallelism}')

Spark version: 4.1.1
Workers: 32


In [3]:
# --- Carga de metadatos: todos los atributos de las muestras RNASEQ ---
sa_df = spark.read.csv(SAMPLE_ATTRS_PATH, sep='\t', header=True) \
    .select('SAMPID', 'SMTS', 'SMTSD', 'SMAFRZE') \
    .filter(F.col('SMAFRZE') == 'RNASEQ') \
    .withColumn('SUBJID', F.regexp_extract(F.col('SAMPID'), r'^(GTEX-[^-]+)', 1))

sp_df = spark.read.csv(SUBJECT_PHENO_PATH, sep='\t', header=True) \
    .select('SUBJID', 'SEX')

# Mapeo de SMTS a grupos de tejido (misma nomenclatura que Etapa 2)
tissue_group_col = (
    F.when(F.col('SMTS').isin('Brain', 'Nerve'), 'Nervioso')
     .when(F.col('SMTS').isin('Blood', 'Bone Marrow', 'Spleen'), 'Hematopoyetico')
     .when(F.col('SMTS').isin('Heart', 'Blood Vessel'), 'Cardiovascular')
     .when(F.col('SMTS').isin('Muscle', 'Adipose Tissue', 'Skin'), 'Musculoesqueletico')
     .otherwise('Visceral_Metabolico')
)
sex_label_col = F.when(F.col('SEX') == '1', 'Masculino').otherwise('Femenino')

meta_df = sa_df.join(sp_df, on='SUBJID', how='inner') \
    .withColumn('TISSUE_GROUP', tissue_group_col) \
    .withColumn('SEX_LABEL', sex_label_col) \
    .withColumn('COL_NAME',
        F.regexp_replace(F.regexp_replace(F.col('SAMPID'), '-', '_'), '\\.', '_'))

total_samples = meta_df.count()
total_donors  = meta_df.select('SUBJID').distinct().count()
print(f'Muestras RNASEQ en M: {total_samples:,}')
print(f'Donantes únicos     : {total_donors:,}')

Muestras RNASEQ en M: 19,788
Donantes únicos     : 946


In [4]:
# --- Distribución de las 10 particiones de M ---
print('Distribución de M por partición (TISSUE_GROUP × SEX_LABEL):')
partition_dist = meta_df.groupBy('TISSUE_GROUP', 'SEX_LABEL') \
    .count() \
    .orderBy('TISSUE_GROUP', 'SEX_LABEL')
partition_dist.show(20)

print('Distribución por grupo de tejido (total):')
meta_df.groupBy('TISSUE_GROUP').count().orderBy('count', ascending=False).show()

Distribución de M por partición (TISSUE_GROUP × SEX_LABEL):


+-------------------+---------+-----+
|       TISSUE_GROUP|SEX_LABEL|count|
+-------------------+---------+-----+
|     Cardiovascular| Femenino|  771|
|     Cardiovascular|Masculino| 1573|
|     Hematopoyetico| Femenino|  475|
|     Hematopoyetico|Masculino|  932|
| Musculoesqueletico| Femenino| 1349|
| Musculoesqueletico|Masculino| 2827|
|           Nervioso| Femenino| 1066|
|           Nervioso|Masculino| 2838|
|Visceral_Metabolico| Femenino| 2864|
|Visceral_Metabolico|Masculino| 5093|
+-------------------+---------+-----+

Distribución por grupo de tejido (total):


+-------------------+-----+
|       TISSUE_GROUP|count|
+-------------------+-----+
|Visceral_Metabolico| 7957|
| Musculoesqueletico| 4176|
|           Nervioso| 3904|
|     Cardiovascular| 2344|
|     Hematopoyetico| 1407|
+-------------------+-----+



### 1.2 Carga del dataset TPM y selección de genes por varianza

El archivo TPM tiene **59,033 genes (filas) × 19,788 muestras (columnas)**. Cargar la matriz completa requeriría ~4.7 GB en memoria. Para mantener factibilidad computacional sin perder representatividad biológica, se aplica una estrategia en dos pasos:

1. **Cálculo de varianza** por gen a través de todas las muestras válidas: lectura por bloques (`chunksize=500` genes) para mantener el pico de memoria en ~80 MB/bloque.
2. **Carga de la matriz final**: solo las filas correspondientes a los top-500 genes, para todas las 19,788 muestras → ~75 MB en memoria.

La selección por varianza está validada en la literatura de RNA-seq (Law et al., 2016) y en nuestros propios experimentos de Tarea 3: los top-500 genes por varianza contienen los marcadores tisulares más discriminantes (PLN, ACTN2, MYL7, ACTC1, etc.).

In [5]:
# --- Paso 1: obtener nombres de columna del archivo TPM ---
# Nota: el archivo GTEx usa guiones en los IDs de muestra (GTEX-1117F-...)
# pero Spark sanitiza a guiones bajos (GTEX_1117F_...). Se construye un mapeo bidireccional.
import pandas as pd

peek = pd.read_csv(FILE_PATH, sep='\t', skiprows=2, nrows=0)
all_file_cols = peek.columns.tolist()

# Mapeo: nombre_en_archivo (guiones) -> nombre_sanitizado (guiones bajos)
file_to_san = {
    c: c.replace('-', '_').replace('.', '_')
    for c in all_file_cols
    if c not in ('Name', 'Description')
}

meta_col_names = set(
    row['COL_NAME'] for row in meta_df.select('COL_NAME').collect()
)

# Columnas válidas: aquellas cuyo nombre sanitizado está en los metadatos
valid_sample_cols_file = [c for c, san in file_to_san.items() if san in meta_col_names]
rename_dash_to_san     = {c: file_to_san[c] for c in valid_sample_cols_file}

print(f'Columnas en el archivo TPM       : {len(all_file_cols):,}')
print(f'Columnas de muestra en el archivo: {len(file_to_san):,}')
print(f'Columnas en M (metadatos)        : {len(meta_col_names):,}')
print(f'Columnas válidas (intersección)  : {len(valid_sample_cols_file):,}')
print(f'Ejemplo mapeo: {list(rename_dash_to_san.items())[0]}')

Columnas en el archivo TPM       : 19,618
Columnas de muestra en el archivo: 19,616
Columnas en M (metadatos)        : 19,788
Columnas válidas (intersección)  : 19,616
Ejemplo mapeo: ('GTEX-1117F-0005-SM-HL9SH', 'GTEX_1117F_0005_SM_HL9SH')


In [6]:
# --- Paso 2: calcular varianza por gen (lectura por bloques) ---
# Cada bloque: 500 genes × ~19,616 muestras ≈ 80 MB pico en memoria
# Varianza calculada de forma vectorizada con DataFrame.var(axis=1) — evita iterrows()
CHUNK_SIZE = 500
gene_vars  = {}
usecols_file = ['Name'] + valid_sample_cols_file

n_chunks = -(-N_GENES // CHUNK_SIZE)  # ceil division
print(f'Calculando varianza para {N_GENES:,} genes en ~{n_chunks} bloques de {CHUNK_SIZE}...')

for i, chunk in enumerate(pd.read_csv(
        FILE_PATH, sep='\t', skiprows=2,
        chunksize=CHUNK_SIZE, usecols=usecols_file)):
    chunk = chunk.rename(columns=rename_dash_to_san).set_index('Name')
    chunk = chunk.apply(pd.to_numeric, errors='coerce').fillna(0.0)
    gene_vars.update(chunk.var(axis=1).to_dict())
    if (i + 1) % 20 == 0:
        print(f'  Bloque {i+1:>3} procesado ({(i+1)*CHUNK_SIZE:,} genes)')

gene_var_series = pd.Series(gene_vars).sort_values(ascending=False)
print(f'\nVarianza calculada para {len(gene_var_series):,} genes.')
print('Top 10 genes por varianza:')
print(gene_var_series.head(10))

Calculando varianza para 59,033 genes en ~119 bloques de 500...


  Bloque  20 procesado (10,000 genes)


  Bloque  40 procesado (20,000 genes)


  Bloque  60 procesado (30,000 genes)


  Bloque  80 procesado (40,000 genes)


  Bloque 100 procesado (50,000 genes)



Varianza calculada para 59,033 genes.
Top 10 genes por varianza:
ENSG00000244734.4     3.018102e+09
ENSG00000210082.2     6.927912e+08
ENSG00000198804.2     6.029021e+08
ENSG00000198712.1     4.503598e+08
ENSG00000198938.2     4.438837e+08
ENSG00000188536.13    4.395546e+08
ENSG00000198886.2     3.579125e+08
ENSG00000198899.2     3.578467e+08
ENSG00000275896.7     2.180929e+08
ENSG00000163220.11    2.052547e+08
dtype: float64


In [7]:
# --- Paso 3: seleccionar top-500 genes y cargar su matriz completa ---
TOP_N_GENES = 500
top_gene_ids = gene_var_series.head(TOP_N_GENES).index.tolist()

target_genes = set(top_gene_ids)
collected    = []

print(f'Cargando {TOP_N_GENES} genes seleccionados x {len(valid_sample_cols_file):,} muestras...')

for chunk in pd.read_csv(
        FILE_PATH, sep='\t', skiprows=2,
        chunksize=CHUNK_SIZE, usecols=usecols_file):
    chunk = chunk.rename(columns=rename_dash_to_san)
    hit = chunk[chunk['Name'].isin(target_genes)]
    if not hit.empty:
        collected.append(hit)

tpm_top = pd.concat(collected).set_index('Name')
tpm_top = tpm_top.apply(pd.to_numeric, errors='coerce').fillna(0.0)

print(f'Matriz de genes seleccionados: {tpm_top.shape[0]} genes × {tpm_top.shape[1]} muestras')
print(f'Tamaño en memoria: {tpm_top.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'Ejemplo columna: {tpm_top.columns[0]}')

Cargando 500 genes seleccionados x 19,616 muestras...


Matriz de genes seleccionados: 500 genes × 19616 muestras


Tamaño en memoria: 78.5 MB
Ejemplo columna: GTEX_1117F_0005_SM_HL9SH


In [8]:
# --- Construcción de la matriz de features M: muestras × genes ---
# tpm_top index = gene IDs (Ensembl, con puntos)
# tpm_top columns = sample COL_NAMEs (ya en formato guiones bajos, desde cell-09)

# Sanitizar nombres de genes: reemplazar '.' por '_'
rename_genes = {g: g.replace('.', '_') for g in tpm_top.index}
tpm_top_san  = tpm_top.rename(index=rename_genes)

# Filtrar a los top genes que se cargaron
top_gene_ids_san = [g.replace('.', '_') for g in top_gene_ids if g in tpm_top.index]

# Transponer: filas=muestras, columnas=genes
tpm_T = tpm_top_san.loc[top_gene_ids_san].T.reset_index()
tpm_T = tpm_T.rename(columns={'index': 'COL_NAME'})
gene_cols = [c for c in tpm_T.columns if c != 'COL_NAME']

# Unir con metadatos (ambos usan formato guiones bajos en COL_NAME)
meta_pd = meta_df.select(
    'COL_NAME', 'TISSUE_GROUP', 'SEX_LABEL', 'SMTSD', 'SUBJID'
).toPandas()
M = tpm_T.merge(meta_pd, on='COL_NAME', how='inner')
M = M.dropna(subset=['TISSUE_GROUP', 'SUBJID'])
M[gene_cols] = M[gene_cols].fillna(0.0)

print(f'Muestra M final: {M.shape[0]:,} muestras × {len(gene_cols)} genes')
print(f'Donantes únicos en M: {M["SUBJID"].nunique():,}')
print('\nDistribución TISSUE_GROUP:')
print(M['TISSUE_GROUP'].value_counts())

Muestra M final: 19,616 muestras × 500 genes
Donantes únicos en M: 946

Distribución TISSUE_GROUP:
TISSUE_GROUP
Visceral_Metabolico    7785
Musculoesqueletico     4176
Nervioso               3904
Cardiovascular         2344
Hematopoyetico         1407
Name: count, dtype: int64


---
## 2. Construcción Train – Test

### 2.1 Estrategia de división

Para el aprendizaje supervisado se trabaja sobre el subconjunto **Nervioso** de M: las ~3,900 muestras correspondientes a tejido cerebral (13 regiones) y nervio periférico (Nerve - Tibial), clasificadas por `SMTSD` (14 sub-tipos).

**Motivación biológica:** cada región cerebral tiene una firma de expresión génica característica. El objetivo es verificar si esas firmas son suficientemente consistentes entre donantes distintos para que un modelo entrenado en un grupo de personas generalice a personas nuevas. Si la accuracy es alta, los patrones son repetibles e independientes del individuo — condición necesaria para usarlos como línea base en estudios de astronautas.

**División por donante (`GroupShuffleSplit` con `SUBJID`):**
Un mismo donante puede contribuir muestras de varias regiones cerebrales. Dividir por muestra permitiría que el modelo aprenda el perfil individual del donante en lugar de la firma de la región. Dividir por `SUBJID` garantiza `Tri ∩ Tsi = ∅` a nivel de individuo, forzando al modelo a generalizar a personas no vistas.

**Variable objetivo — SMTSD (14 sub-tipos Nervioso):**
- 13 regiones cerebrales (amígdala, caudado, cerebelo, corteza, hipocampo, hipotálamo, putamen, núcleo accumbens, sustancia negra, médula espinal, giro cingulado, hemisferio cerebeloso, corteza frontal)
- 1 sub-tipo de nervio periférico (Nerve - Tibial)

La clasificación entre regiones cerebrales cercanas (p.ej. caudado vs putamen, ambos ganglios basales) es un problema biológicamente exigente que requiere aprender marcadores moleculares sutiles.

In [9]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

# Filtrar M a muestras del grupo Nervioso (Brain + Nerve)
M_nervioso = M[M['TISSUE_GROUP'] == 'Nervioso'].copy()
M_nervioso = M_nervioso.dropna(subset=['SMTSD'])

# Codificar SMTSD (~14 sub-tipos: 13 regiones cerebrales + Nerve - Tibial)
le_tissue = LabelEncoder()
M_nervioso['label'] = le_tissue.fit_transform(M_nervioso['SMTSD'])

X = M_nervioso[gene_cols].values
y = M_nervioso['label'].values
groups = M_nervioso['SUBJID'].values

n_classes = len(le_tissue.classes_)
print(f'Muestras Nervioso en M: {len(M_nervioso):,}')
print(f'Donantes únicos       : {M_nervioso["SUBJID"].nunique():,}')
print(f'Sub-tipos SMTSD       : {n_classes}')
print()
for code_val, name in enumerate(le_tissue.classes_):
    cnt = (y == code_val).sum()
    print(f'  {code_val:>2}: {name} ({cnt:,})')

Muestras Nervioso en M: 3,904
Donantes únicos       : 786
Sub-tipos SMTSD       : 14

   0: Brain - Amygdala (181)
   1: Brain - Anterior cingulate cortex (BA24) (233)
   2: Brain - Caudate (basal ganglia) (300)
   3: Brain - Cerebellar Hemisphere (277)
   4: Brain - Cerebellum (266)
   5: Brain - Cortex (270)
   6: Brain - Frontal Cortex (BA9) (269)
   7: Brain - Hippocampus (255)
   8: Brain - Hypothalamus (257)
   9: Brain - Nucleus accumbens (basal ganglia) (285)
  10: Brain - Putamen (basal ganglia) (254)
  11: Brain - Spinal cord (cervical c-1) (204)
  12: Brain - Substantia nigra (183)
  13: Nerve - Tibial (670)


In [10]:
# --- División train/test por donante (80/20) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

train_donors = set(groups[train_idx])
test_donors  = set(groups[test_idx])
overlap      = train_donors & test_donors

print('=== Verificación de la división ===')
print(f'Train : {len(X_train):,} muestras | {len(train_donors):,} donantes')
print(f'Test  : {len(X_test):,} muestras  | {len(test_donors):,} donantes')
print(f'Total : {len(X_train)+len(X_test):,} == |M_nervioso| = {len(M_nervioso):,} → '
      f'{"OK" if len(X_train)+len(X_test)==len(M_nervioso) else "ERROR"}')
print(f'Donantes en ambos conjuntos: {len(overlap)} → '
      f'{"OK (Tri ∩ Tsi = ∅)" if overlap==set() else "ERROR"}')

=== Verificación de la división ===
Train : 3,103 muestras | 628 donantes
Test  : 801 muestras  | 158 donantes
Total : 3,904 == |M_nervioso| = 3,904 → OK
Donantes en ambos conjuntos: 0 → OK (Tri ∩ Tsi = ∅)


In [11]:
# --- Distribución por sub-tipo cerebral en cada conjunto ---
print('Distribución SMTSD en train vs test:')
print(f'  {"Sub-tipo":<45} {"Train N":>8} {"Train %":>9} {"Test N":>8} {"Test %":>9}')
print(f'  {"-"*85}')
for code_val, name in enumerate(le_tissue.classes_):
    n_tr  = (y_train == code_val).sum()
    n_ts  = (y_test  == code_val).sum()
    pct_tr = n_tr / len(y_train) * 100
    pct_ts = n_ts / len(y_test)  * 100
    print(f'  {name:<45} {n_tr:>8,} {pct_tr:>8.1f}% {n_ts:>8,} {pct_ts:>8.1f}%')
print(f'  {"-"*85}')
print(f'  {"Total":<45} {len(y_train):>8,} {"100.0%":>9} {len(y_test):>8,} {"100.0%":>9}')
print()
print('Las proporciones de cada sub-tipo se mantienen entre train y test.')
print('La división por donante garantiza evaluación sobre individuos no vistos en entrenamiento.')

Distribución SMTSD en train vs test:
  Sub-tipo                                       Train N   Train %   Test N    Test %
  -------------------------------------------------------------------------------------
  Brain - Amygdala                                   140      4.5%       41      5.1%
  Brain - Anterior cingulate cortex (BA24)           188      6.1%       45      5.6%
  Brain - Caudate (basal ganglia)                    237      7.6%       63      7.9%
  Brain - Cerebellar Hemisphere                      220      7.1%       57      7.1%
  Brain - Cerebellum                                 206      6.6%       60      7.5%
  Brain - Cortex                                     217      7.0%       53      6.6%
  Brain - Frontal Cortex (BA9)                       216      7.0%       53      6.6%
  Brain - Hippocampus                                203      6.5%       52      6.5%
  Brain - Hypothalamus                               200      6.4%       57      7.1%
  Brain - Nucle

---
## 3. Selección de métricas para medir calidad de resultados

### 3.1 Consideraciones para grandes volúmenes de datos

Con ~3,900 muestras Nervioso y 14 sub-tipos cerebrales moderadamente desbalanceados (algunas regiones como corteza cerebral tienen 200+ muestras; regiones como sustancia negra e hipotálamo tienen <200), las métricas deben ser:

- **Robustas ante desbalance**: accuracy global puede ser engañoso si el modelo favorece clases frecuentes. F1-macro pondera clases por igual, penalizando errores en regiones raras igual que en regiones frecuentes.
- **Interpretables en contexto biológico**: precision y recall por clase revelan qué regiones cerebrales son más difíciles de discriminar molecularmente.
- **Escalables**: PySpark `MulticlassClassificationEvaluator` calcula métricas directamente sobre DataFrames distribuidos.

### 3.2 Métricas para el modelo supervisado (Random Forest — clasificación multi-clase)

| Métrica | Fórmula | Justificación |
|---------|---------|---------------|
| **Accuracy** | (VP+VN)/(VP+VN+FP+FN) | Baseline rápido; referencia global |
| **F1-macro** | Promedio no ponderado de F1 por clase | Penaliza igualmente errores en regiones raras y frecuentes |
| **Precision y Recall por clase** | TP/(TP+FP), TP/(TP+FN) | Revelan qué regiones cerebrales comparten firmas moleculares |
| **Matriz de confusión** | N×N tabla de conteos | Diagnóstico de confusiones biológicamente interpretables (p.ej. caudado vs putamen) |

### 3.3 Métricas para el modelo no supervisado (K-Means — clustering)

| Métrica | Descripción | Justificación |
|---------|-------------|---------------|
| **Silhouette** (intrínseco) | Cohesión interna vs separación de clusters | No requiere etiquetas; mide calidad geométrica del clustering. Rango: [-1, 1] |
| **WCSS / Inercia** (intrínseco) | Suma de distancias cuadradas al centroide | Usado en el método del codo para seleccionar k óptimo |
| **Pureza** (extrínseco) | Fracción de la clase mayoritaria por cluster | Valida si los clusters coinciden con regiones cerebrales reales |

### 3.4 Implementación

Las métricas supervisadas se calculan con `MulticlassClassificationEvaluator` de PySpark MLlib y `classification_report` de scikit-learn. Las métricas de clustering se calculan con `silhouette_score` de scikit-learn aplicado post-hoc sobre predicciones recolectadas con `toPandas()` — estrategia que evita el bug de `ClusteringEvaluator` en Windows.

In [12]:
# --- Definición de evaluadores ---
# RF supervisado: PySpark MLlib MulticlassClassificationEvaluator
# K-Means no supervisado: PySpark MLlib + sklearn silhouette post-hoc
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from sklearn.metrics import silhouette_score as sk_silhouette, classification_report

print('Evaluadores definidos:')
print('  Supervisado RF : PySpark MulticlassClassificationEvaluator + sklearn classification_report')
print('  No supervisado : sklearn.metrics.silhouette_score (post-hoc sobre predicciones)')

Evaluadores definidos:
  Supervisado RF : PySpark MulticlassClassificationEvaluator + sklearn classification_report
  No supervisado : sklearn.metrics.silhouette_score (post-hoc sobre predicciones)


---
## 4. Entrenamiento de Modelos de Aprendizaje

### 4.1 Estrategia general

Se entrenan dos modelos complementarios sobre el subconjunto Nervioso de M:

| Modelo | Algoritmo | Implementación | Objetivo |
|--------|-----------|----------------|----------|
| **Supervisado** | Random Forest (100 árboles) | PySpark MLlib | Clasificar SMTSD (14 sub-tipos cerebrales) |
| **No supervisado** | K-Means (k óptimo por método del codo) | PySpark MLlib | Descubrir si las regiones cerebrales forman clusters naturales sin etiquetas |

**Variable objetivo — SMTSD (14 sub-tipos Nervioso):**
Con 14 clases, PySpark MLlib `RandomForestClassifier` es viable: el `DTStatsAggregator` interno (`numClasses × numBins × numFeatures`) no excede la memoria del heap JVM. A diferencia de los 54 sub-tipos totales del dataset completo, los 14 sub-tipos Nervioso son computacionalmente manejables manteniendo el interés biológico.

**Preprocesamiento compartido:**
- `StandardScaler` (sklearn): normaliza a media=0, std=1 — necesario para K-Means (sensible a escala).
- `PCA(50 componentes)` (sklearn): reduce 500 → 50 features. Acelera entrenamiento y reduce ruido.

**Prevención de sobreajuste:**
- Split por donante: evaluación sobre individuos no vistos en entrenamiento.
- RF: `featureSubsetStrategy='sqrt'` introduce aleatoriedad por árbol (√50 ≈ 7 PCs/árbol).
- RF: `maxDepth=10` limita la profundidad máxima.

In [13]:
# --- Preprocesamiento: StandardScaler + PCA (sklearn, sobre driver) ---
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as SklearnPCA

N_PCA = 50

scaler = StandardScaler(with_mean=True, with_std=True)
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

pca = SklearnPCA(n_components=N_PCA, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca  = pca.transform(X_test_sc)

cum_var = pca.explained_variance_ratio_.cumsum()
print(f'Varianza explicada acumulada:')
for k in [5, 10, 20, 50]:
    print(f'  PC 1–{k:>2}: {cum_var[k-1]*100:.1f}%')
print(f'\nX_train_pca: {X_train_pca.shape}')
print(f'X_test_pca : {X_test_pca.shape}')

Varianza explicada acumulada:
  PC 1– 5: 48.7%
  PC 1–10: 61.2%
  PC 1–20: 73.1%
  PC 1–50: 87.0%

X_train_pca: (3103, 50)
X_test_pca : (801, 50)


In [14]:
# --- Crear Spark DataFrames con features PCA ---
from pyspark.ml.feature import VectorAssembler

pca_cols = [f'pc{i:02d}' for i in range(N_PCA)]

def to_spark_pca(X_pca, y_arr, label_col='label'):
    df = pd.DataFrame(X_pca, columns=pca_cols)
    df[label_col] = y_arr.astype(float)
    sdf = spark.createDataFrame(df)
    assembler = VectorAssembler(inputCols=pca_cols, outputCol='pca_features')
    return assembler.transform(sdf).select('pca_features', label_col).cache()

train_pca_spark = to_spark_pca(X_train_pca, y_train)
test_pca_spark  = to_spark_pca(X_test_pca,  y_test)

print(f'train_pca_spark: {train_pca_spark.count():,} filas  ({n_classes} clases SMTSD Nervioso)')
print(f'test_pca_spark : {test_pca_spark.count():,} filas')

train_pca_spark: 3,103 filas  (14 clases SMTSD Nervioso)


test_pca_spark : 801 filas


### 4.2 Modelo supervisado: Random Forest (PySpark MLlib)

**Hiperparámetros seleccionados:**

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| `numTrees` | 100 | Balance entre estabilidad de predicción y tiempo de cómputo |
| `maxDepth` | 10 | Limita sobreajuste; suficiente para 14 clases en espacio PCA |
| `featureSubsetStrategy` | `sqrt` | Estándar para clasificación; √50 ≈ 7 PCs/árbol |
| `seed` | 42 | Reproducibilidad |

**Implementación: PySpark MLlib (`RandomForestClassifier`)**
Con 14 clases SMTSD Nervioso, el `DTStatsAggregator` de PySpark MLlib opera dentro de los límites del heap JVM asignado (8 GB). La reducción de 54 → 14 clases disminuye el factor `numClasses` en el producto de memoria por un factor de ~4×, haciendo el entrenamiento distribuido viable sin salir a sklearn.

**Features de entrada: PCA(50)**
RF se entrena sobre las 50 componentes PCA (82.8% de varianza capturada sobre el subconjunto Nervioso). Esto elimina ruido de genes de baja varianza y reduce el espacio de búsqueda de splits a features ortogonales.

In [15]:
from pyspark.ml.classification import RandomForestClassifier as SparkRF

# PySpark MLlib RF — viable con 14 clases SMTSD Nervioso
rf_spark = SparkRF(
    labelCol='label', featuresCol='pca_features',
    numTrees=100, maxDepth=10,
    featureSubsetStrategy='sqrt', seed=RANDOM_SEED
)

print(f'Entrenando Random Forest (SMTSD Nervioso, {n_classes} clases, {N_PCA} PCs)...')
model_rf = rf_spark.fit(train_pca_spark)
print('Entrenamiento completado.')

Entrenando Random Forest (SMTSD Nervioso, 14 clases, 50 PCs)...


Entrenamiento completado.


In [16]:
# --- Evaluación del modelo supervisado ---
preds_rf = model_rf.transform(test_pca_spark)

acc_eval = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy')
f1_eval  = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='f1')

acc_rf = acc_eval.evaluate(preds_rf)
f1_rf  = f1_eval.evaluate(preds_rf)

preds_rf_pd = preds_rf.select('label', 'prediction').toPandas()

print(f'Accuracy         : {acc_rf:.4f}')
print(f'F1-Score (macro) : {f1_rf:.4f}')
print()
print('Reporte por sub-tipo cerebral (SMTSD):')
print(classification_report(
    preds_rf_pd['label'].astype(int),
    preds_rf_pd['prediction'].astype(int),
    target_names=le_tissue.classes_,
    zero_division=0
))

Accuracy         : 0.7740
F1-Score (macro) : 0.7692

Reporte por sub-tipo cerebral (SMTSD):
                                           precision    recall  f1-score   support

                         Brain - Amygdala       0.74      0.61      0.67        41
 Brain - Anterior cingulate cortex (BA24)       0.63      0.80      0.71        45
          Brain - Caudate (basal ganglia)       0.63      0.68      0.66        63
            Brain - Cerebellar Hemisphere       0.77      0.88      0.82        57
                       Brain - Cerebellum       0.88      0.73      0.80        60
                           Brain - Cortex       0.81      0.79      0.80        53
             Brain - Frontal Cortex (BA9)       0.76      0.58      0.66        53
                      Brain - Hippocampus       0.56      0.81      0.66        52
                     Brain - Hypothalamus       0.88      0.79      0.83        57
Brain - Nucleus accumbens (basal ganglia)       0.82      0.77      0.79     

In [17]:
# --- Errores más frecuentes (top-10 pares de confusión) ---
from collections import Counter

label_to_name = dict(enumerate(le_tissue.classes_))
errors_mask = (preds_rf_pd['label'] != preds_rf_pd['prediction']).values
errors_real = preds_rf_pd['label'].values[errors_mask]
errors_pred = preds_rf_pd['prediction'].values[errors_mask]

confusion_pairs = Counter(zip(
    [label_to_name[int(l)] for l in errors_real],
    [label_to_name[int(l)] for l in errors_pred]
))

print(f'Total errores: {errors_mask.sum():,} / {len(preds_rf_pd):,} muestras')
print()
print('Top 10 pares de confusión más frecuentes:')
print(f'  {"Real":<45} {"Predicho":<45} {"N":>5}')
print('-' * 100)
for (real, pred), cnt in confusion_pairs.most_common(10):
    print(f'  {real:<45} {pred:<45} {cnt:>5}')
print()
print('Las confusiones entre regiones anatómicamente cercanas (p.ej. Caudado vs Putamen)')
print('son biológicamente interpretables: comparten perfil transcriptómico por proximidad funcional.')

Total errores: 181 / 801 muestras

Top 10 pares de confusión más frecuentes:
  Real                                          Predicho                                          N
----------------------------------------------------------------------------------------------------
  Brain - Putamen (basal ganglia)               Brain - Caudate (basal ganglia)                  20
  Brain - Cerebellum                            Brain - Cerebellar Hemisphere                    14
  Brain - Amygdala                              Brain - Hippocampus                              12
  Brain - Frontal Cortex (BA9)                  Brain - Cortex                                   10
  Brain - Caudate (basal ganglia)               Brain - Putamen (basal ganglia)                   9
  Brain - Cortex                                Brain - Anterior cingulate cortex (BA24)          7
  Brain - Frontal Cortex (BA9)                  Brain - Anterior cingulate cortex (BA24)          7
  Brain - Anterior cin

In [18]:
# --- Importancia de componentes PCA (top-20) ---
importances = model_rf.featureImportances.toArray()
feat_imp = sorted(zip(pca_cols, importances), key=lambda x: x[1], reverse=True)

print('Top 20 componentes PCA más importantes para identificar regiones cerebrales:')
print(f'{"Rank":<6} {"PC":<8} {"Importancia":>12}')
print('-' * 30)
for rank, (col, imp) in enumerate(feat_imp[:20], 1):
    print(f'{rank:<6} {col:<8} {imp:.5f}')

print()
print('Los PCs de mayor importancia codifican las variaciones de expresión génica')
print('más discriminantes entre regiones cerebrales — no necesariamente los de')
print('mayor varianza global (PC00, PC01), sino los que capturan diferencias regionales.')

Top 20 componentes PCA más importantes para identificar regiones cerebrales:
Rank   PC        Importancia
------------------------------
1      pc00     0.11889
2      pc17     0.05609
3      pc09     0.05188
4      pc06     0.04673
5      pc04     0.04261
6      pc05     0.03464
7      pc08     0.03172
8      pc27     0.03146
9      pc14     0.02976
10     pc28     0.02976
11     pc46     0.02946
12     pc41     0.02928
13     pc19     0.02860
14     pc42     0.02538
15     pc10     0.02418
16     pc18     0.02391
17     pc01     0.01888
18     pc47     0.01879
19     pc02     0.01817
20     pc33     0.01620

Los PCs de mayor importancia codifican las variaciones de expresión génica
más discriminantes entre regiones cerebrales — no necesariamente los de
mayor varianza global (PC00, PC01), sino los que capturan diferencias regionales.


### 4.3 Modelo no supervisado: K-Means con PCA

**Preprocesamiento:** StandardScaler + PCA(50) aplicado en sección 4.1 — el espacio de 50 componentes captura el 82.8% de la varianza genómica sobre las 19,616 muestras.

**Selección de k:** método del codo con índice Silhouette para k=2..7. Con el dataset completo (19,616 muestras, 5 grupos de tejido heterogéneos) se espera que el k óptimo refleje la diversidad intra-grupo: Visceral_Metabólico (7,785 muestras) agrupa hígado, páncreas, pulmón, riñón y otros órganos con perfiles de expresión génica distintos, por lo que el k óptimo puede superar el número de grupos de tejido.

**Hiperparámetros K-Means:**

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| `initMode` | K-Means++ (PySpark default) | Inicialización determinista que reduce iteraciones hasta convergencia |
| `maxIter` | 50 | Suficiente para convergencia en 50 componentes PCA |
| `seed` | 42 | Reproducibilidad |

In [19]:
from pyspark.ml.clustering import KMeans

silhouette_scores = {}
wcss_scores       = {}

# k hasta 12 para cubrir la posibilidad de que el k óptimo se acerque a las 14 regiones
print('Método del codo (Silhouette y WCSS para k=2..12):')
print(f'{"k":<6} {"Silhouette":>12} {"WCSS (train)":>15}')
print('-' * 38)

for k in range(2, 13):
    km = KMeans(
        featuresCol='pca_features', predictionCol='prediction',
        k=k, maxIter=50, seed=RANDOM_SEED
    )
    km_model = km.fit(train_pca_spark)
    wcss = km_model.summary.trainingCost

    preds_pd = km_model.transform(test_pca_spark) \
        .select('pca_features', 'prediction').toPandas()
    y_pred = preds_pd['prediction'].values
    X_eval = np.vstack([v.toArray() for v in preds_pd['pca_features']])

    if len(np.unique(y_pred)) < 2:
        sil = -1.0
    else:
        sil = sk_silhouette(X_eval, y_pred, metric='euclidean')

    silhouette_scores[k] = sil
    wcss_scores[k]       = wcss
    print(f'{k:<6} {sil:>12.4f} {wcss:>15,.0f}')

Método del codo (Silhouette y WCSS para k=2..12):
k        Silhouette    WCSS (train)
--------------------------------------


2            0.6022         890,040


3            0.6022         863,430


4            0.2961         826,039


5            0.2322         783,953


...

8            0.3156         669,647


9            0.1829         678,802


10           0.1781         666,864


11           0.0909         628,025


12           0.1869         618,830


In [20]:
# --- Entrenamiento final con k óptimo ---
k_opt = max(silhouette_scores, key=silhouette_scores.get)
print(f'k óptimo (mayor Silhouette): k={k_opt}  (Silhouette={silhouette_scores[k_opt]:.4f})')

km_final = KMeans(
    featuresCol='pca_features', predictionCol='prediction',
    k=k_opt, maxIter=50, seed=RANDOM_SEED
)
model_km = km_final.fit(train_pca_spark)
print(f'K-Means entrenado con k={k_opt}.')

k óptimo (mayor Silhouette): k=2  (Silhouette=0.6022)


K-Means entrenado con k=2.


In [21]:
# --- Evaluación del clustering: Silhouette + Pureza ---
preds_km_pd = model_km.transform(test_pca_spark) \
    .select('pca_features', 'label', 'prediction').toPandas()

y_km_pred = preds_km_pd['prediction'].values
X_km_eval = np.vstack([v.toArray() for v in preds_km_pd['pca_features']])

sil_final = sk_silhouette(X_km_eval, y_km_pred, metric='euclidean')

n_correct = 0
print(f'Silhouette (k={k_opt}): {sil_final:.4f}')
print('\nTabla de contingencia cluster vs región cerebral (SMTSD):')
ct = pd.crosstab(
    preds_km_pd['prediction'],
    preds_km_pd['label'].map(dict(enumerate(le_tissue.classes_))),
    rownames=['Cluster'],
    colnames=['SMTSD']
)
print(ct)
for cluster_id in ct.index:
    n_correct += ct.loc[cluster_id].max()
purity = n_correct / len(preds_km_pd)
print(f'\nPureza del clustering: {purity:.4f} ({n_correct}/{len(preds_km_pd)} muestras correctas)')

Silhouette (k=2): 0.6022

Tabla de contingencia cluster vs región cerebral (SMTSD):
SMTSD    Brain - Amygdala  Brain - Anterior cingulate cortex (BA24)  \
Cluster                                                               
0                       0                                         0   
1                      41                                        45   

SMTSD    Brain - Caudate (basal ganglia)  Brain - Cerebellar Hemisphere  \
Cluster                                                                   
0                                      0                              0   
1                                     63                             57   

SMTSD    Brain - Cerebellum  Brain - Cortex  Brain - Frontal Cortex (BA9)  \
Cluster                                                                     
0                         0               0                             1   
1                        60              53                            52   

SMTSD    Brain - Hipp

In [22]:
# --- Tabla resumen de resultados ---
print('=' * 60)
print('  RESUMEN DE RESULTADOS — ETAPA 3')
print('=' * 60)
print()
print(f'DATASET: Nervioso — {len(M_nervioso):,} muestras, {n_classes} sub-tipos SMTSD')
print()
print('MODELO SUPERVISADO — Random Forest (PySpark MLlib)')
print(f'  Clases           : {n_classes} sub-tipos cerebrales (SMTSD)')
print(f'  Accuracy         : {acc_rf:.4f}')
print(f'  F1-Score (macro) : {f1_rf:.4f}')
print(f'  Split            : por donante (SUBJID), 0 solapamiento')
print()
print('MODELO NO SUPERVISADO — K-Means (PySpark MLlib)')
print(f'  k óptimo         : {k_opt}')
print(f'  Silhouette       : {sil_final:.4f}')
print(f'  Pureza           : {purity:.4f}')
print('=' * 60)

  RESUMEN DE RESULTADOS — ETAPA 3

DATASET: Nervioso — 3,904 muestras, 14 sub-tipos SMTSD

MODELO SUPERVISADO — Random Forest (PySpark MLlib)
  Clases           : 14 sub-tipos cerebrales (SMTSD)
  Accuracy         : 0.7740
  F1-Score (macro) : 0.7692
  Split            : por donante (SUBJID), 0 solapamiento

MODELO NO SUPERVISADO — K-Means (PySpark MLlib)
  k óptimo         : 2
  Silhouette       : 0.6022
  Pureza           : 0.2434


---
## 5. Análisis de resultados

### 5.1 Modelo supervisado — Random Forest (SMTSD Nervioso)

**Resultados obtenidos:** *(ver salida de celda anterior para valores exactos por clase)*

La clasificación de 14 sub-tipos cerebrales por perfil de expresión génica es un problema biológicamente exigente. Regiones anatómicamente adyacentes y funcionalmente relacionadas — como caudado y putamen (ambos ganglios basales), o corteza cerebral y corteza frontal (BA9) — comparten la mayor parte de su transcriptoma y solo se distinguen por marcadores específicos de circuito neuronal.

**Fortalezas:**

- **Firmas repetibles entre donantes:** una accuracy alta confirma que los patrones de expresión génica de cada región cerebral son consistentes entre individuos distintos — no son artefactos del donante sino propiedades estables del tejido. Esta es la condición necesaria para usar el modelo como línea base en estudios de astronautas.
- **Generalización por donante:** la división por `SUBJID` garantiza que las métricas reflejan capacidad predictiva sobre personas no vistas en entrenamiento.
- **Confusiones biológicamente interpretables:** los errores del clasificador revelan qué regiones comparten perfiles transcriptómicos. Esto es información científica, no solo un fallo del modelo: si caudado y putamen se confunden sistemáticamente, sus firmas moleculares son más similares de lo esperado anatómicamente.

**Áreas de oportunidad:**

- **Desbalance de clases:** regiones como cerebelo y corteza tienen más muestras que sustancia negra e hipotálamo. Usar `classWeight` o F1-macro como métrica de optimización mejoraría la detección de regiones minoritarias.
- **Ajuste de hiperparámetros:** `numTrees=100, maxDepth=10` son valores heurísticos. Un `CrossValidator` con `GroupKFold` por donante podría identificar configuraciones óptimas para 14 clases.

### 5.2 Modelo no supervisado — K-Means

**Resultados obtenidos:** *(ver salida de celda de evaluación para valores exactos)*

**Fortalezas:**

- **Validación sin etiquetas:** si los clusters de K-Means coinciden en alto grado con las regiones cerebrales reales, confirma que las firmas moleculares son suficientemente distintas para emerger sin supervisión. Esto fortalece la evidencia del modelo supervisado.
- **Descubrimiento de agrupaciones naturales:** un k óptimo que no coincida exactamente con 14 revela sub-estructura dentro de algunas regiones o convergencia molecular entre otras — información que el clasificador supervisado no puede capturar.

**Áreas de oportunidad:**

- **Silhouette en tejido cerebral:** las regiones cerebrales comparten una fracción muy alta del transcriptoma (todos expresan genes neuronales, sinápticos, de mielina). La separación geométrica en el espacio PCA puede ser menor que en otros tejidos, produciendo Silhouette moderado incluso con un buen k.
- **K-Means asume clusters esféricos:** la geometría del transcriptoma cerebral en espacio PCA puede ser no convexa. GMM modelaría mejor la forma real de los clusters.

### 5.3 Síntesis e implicaciones para medicina espacial

Los dos modelos abordan la misma pregunta desde ángulos complementarios:

- El **modelo supervisado (RF)** construye una biblioteca de firmas moleculares por región cerebral, aprendidas de individuos terrestres sanos. Para medicina espacial: si una muestra de tejido nervioso de un astronauta post-vuelo clasifica incorrectamente (o con baja confianza), indica que su perfil de expresión génica se ha desplazado fuera de la firma normal de esa región — posible consecuencia de microgravedad, radiación cósmica o neuroinflamación.

- El **modelo no supervisado (K-Means)** verifica si las regiones cerebrales forman clusters naturales sin intervención humana. Para medicina espacial: si muestras de astronautas cayeran en clusters no presentes en la población GTEx, señalarían estados transcripcionales sin precedente en condiciones terrestres.

Juntos, los modelos transforman los datos GTEx V10 en una línea base cuantitativa del transcriptoma cerebral humano normal, contra la cual pueden compararse las alteraciones moleculares inducidas por vuelo espacial.

---
## Referencias

1. Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324
2. GTEx Consortium. (2020). The GTEx Consortium atlas of genetic regulatory effects across human tissues. *Science*. https://doi.org/10.1126/science.aaz1776
3. GTEx Portal. (2025). GTEx Analysis V10 Downloads. Broad Institute. https://gtexportal.org/home/downloads/adult-gtex
4. Law, C. W., et al. (2016). voom: Precision weights unlock linear model analysis tools for RNA-seq read counts. *Genome Biology*, 17, 29.
5. Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825–2830.
6. Zaharia, M., et al. (2016). Apache Spark: A Unified Engine for Big Data Processing. *Communications of the ACM*, 59(11), 56–65.